In [1]:
##!pip install pdfplumber reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 10.6 MB/s  0:00:00 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 10.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 11.4 MB/s  0:00:001.9 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 10.9 MB/s  0:00:001.5 MB/s eta 0:00:01
  Attempting uninstall: Pillow
    Found existing installation: pillow 12.0.0
    Uninstalling pillow-12.0.0:
      Successfully uninstalled pillow-12.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pdfplumber] 3/5 [pdfminer.six]


In [2]:
import os
import pdfplumber
from pathlib import Path
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

RAW_DIR = Path("data/raw_invoices")
TEXT_DIR = Path("data/processed/raw_text")
RAW_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
## Generated sample German invoices (synthetic test data)

def create_sample_invoice(filepath, vendor, invoice_number, date,
                           net_amount, vat_rate, currency="EUR"):
    vat_amount = round(net_amount * vat_rate / 100, 2)
    total = round(net_amount + vat_amount, 2)

    c = canvas.Canvas(str(filepath), pagesize=A4)
    c.setFont("Helvetica-Bold", 14)
    c.drawString(50, 800, f"RECHNUNG / INVOICE")

    c.setFont("Helvetica", 11)
    lines = [
        f"Rechnungssteller: {vendor}",
        f"Rechnungsnummer: {invoice_number}",
        f"Rechnungsdatum: {date}",
        "",
        f"Nettobetrag: {net_amount:.2f} {currency}",
        f"USt-Satz: {vat_rate}%",
        f"USt-Betrag: {vat_amount:.2f} {currency}",
        f"Gesamtbetrag: {total:.2f} {currency}",
    ]

    y = 770
    for line in lines:
        c.drawString(50, y, line)
        y -= 20

    c.save()
    print(f"Created: {filepath}")


# Generate 3 sample invoices
create_sample_invoice(RAW_DIR / "invoice_001.pdf", "Buerobedarf Mueller GmbH",
                       "RE-2026-0451", "05.07.2026", 250.00, 19)

create_sample_invoice(RAW_DIR / "invoice_002.pdf", "Software Solutions AG",
                       "INV-9981", "10.07.2026", 1200.00, 19)

create_sample_invoice(RAW_DIR / "invoice_003.pdf", "Bio-Catering Berlin",
                       "BC-2026-077", "12.07.2026", 85.50, 7)

Created: data/raw_invoices/invoice_001.pdf
Created: data/raw_invoices/invoice_002.pdf
Created: data/raw_invoices/invoice_003.pdf


In [4]:
##  Extracted raw text from PDFs

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text.strip()


def process_all_invoices():
    results = {}
    pdf_files = list(RAW_DIR.glob("*.pdf"))

    if not pdf_files:
        print("No PDF files found in", RAW_DIR)
        return results

    for pdf_file in pdf_files:
        text = extract_text_from_pdf(pdf_file)
        results[pdf_file.name] = text

        # Saved raw text alongside for the next step (field extraction)
        out_path = TEXT_DIR / f"{pdf_file.stem}.txt"
        out_path.write_text(text, encoding="utf-8")

        print(f"--- {pdf_file.name} ---")
        print(text)
        print()

    return results


extracted_texts = process_all_invoices()

--- invoice_001.pdf ---
RECHNUNG / INVOICE
Rechnungssteller: Buerobedarf Mueller GmbH
Rechnungsnummer: RE-2026-0451
Rechnungsdatum: 05.07.2026
Nettobetrag: 250.00 EUR
USt-Satz: 19%
USt-Betrag: 47.50 EUR
Gesamtbetrag: 297.50 EUR

--- invoice_002.pdf ---
RECHNUNG / INVOICE
Rechnungssteller: Software Solutions AG
Rechnungsnummer: INV-9981
Rechnungsdatum: 10.07.2026
Nettobetrag: 1200.00 EUR
USt-Satz: 19%
USt-Betrag: 228.00 EUR
Gesamtbetrag: 1428.00 EUR

--- invoice_003.pdf ---
RECHNUNG / INVOICE
Rechnungssteller: Bio-Catering Berlin
Rechnungsnummer: BC-2026-077
Rechnungsdatum: 12.07.2026
Nettobetrag: 85.50 EUR
USt-Satz: 7%
USt-Betrag: 5.99 EUR
Gesamtbetrag: 91.49 EUR



In [5]:
## Checking again

print(f"Processed {len(extracted_texts)} invoice(s)")
print(f"Raw text files saved to: {TEXT_DIR.resolve()}")

Processed 3 invoice(s)
Raw text files saved to: /Users/akashkumarsamantray/invoice-data-extractor-ai/data/processed/raw_text


In [ ]:
## Step 1: PDF Text Extraction (`01_extract_text.ipynb`)

### What this notebook does
This is the first stage of the invoice extraction pipeline. It converts raw PDF invoices into plain text, which later steps use to pull out structured data (vendor, VAT, totals, etc.).

### Process
1. **Generate sample invoices** — since no real invoice data was available yet, 3 synthetic German invoices were created using `reportlab`, each with realistic fields (Rechnungsnummer, Rechnungsdatum, Nettobetrag, USt-Satz, USt-Betrag, Gesamtbetrag) and different VAT rates (19% and 7%) to test both standard cases.
2. **Extract text from PDFs** — each PDF is opened with `pdfplumber` and its text content is extracted page by page.
3. **Save raw text** — the extracted text for each invoice is saved as a `.txt` file in `data/processed/raw_text/`, so the next step (LLM-based field extraction) can work directly from clean text instead of re-parsing PDFs every time.

### Why PDF → text first, instead of PDF → LLM directly
Separating text extraction from field extraction keeps the pipeline modular:
- Text extraction can be tested and debugged independently of the AI step
- Raw text is cached to disk, so re-running the extraction step doesn't cost API calls
- If a different document type is added later (e.g. scanned invoices needing OCR), only this step needs to change

### Output
- `data/raw_invoices/` — 3 sample PDF invoices
- `data/processed/raw_text/` — matching `.txt` files with extracted text

### Next step
`02_extract_fields.ipynb` — send the raw text to the Claude API with a structured JSON prompt to extract vendor, invoice number, date, VAT rate, and totals into clean structured data.